# Fraud Detection on an Imbalanced Transactions Dataset
**Oasis Infobyte SIP · Data Analytics · Level 2 · Task 3**

**Goal:** detect fraudulent card transactions where fraud is extremely rare, treating class imbalance as the core challenge.

**Dataset:** Kaggle *Credit Card Fraud Detection* (`creditcard.csv`, 284,807 transactions, 492 frauds). Put `creditcard.csv` next to this notebook. If the file is missing, the notebook falls back to a **synthetic stand-in** so the code still runs. Re-run with the real file for your final results and PNGs.

In [ ]:
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
                             precision_score, recall_score, f1_score, accuracy_score)
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RS = 42
os.makedirs("images", exist_ok=True)
def save(name):
    plt.tight_layout(); plt.savefig(f"images/{name}.png", dpi=150, bbox_inches="tight"); plt.show()

## 1. Load data

In [ ]:
def make_synthetic(n=60000, fraud_rate=0.0035, seed=RS):
    """Stand-in with the same schema as creditcard.csv. NOT real data."""
    r = np.random.default_rng(seed)
    y = (r.random(n) < fraud_rate).astype(int)
    X = r.normal(size=(n, 28))
    f = y == 1
    for j, shift in {3: 2.5, 10: 2.0, 11: -2.5, 13: -3.0, 9: -2.0, 16: -2.5}.items():
        X[f, j] += shift
    t = np.sort(r.integers(0, 172800, n))
    amt = np.where(f, r.lognormal(3.8, 1.5, n), r.lognormal(3.5, 1.3, n))
    df = pd.DataFrame(X, columns=[f"V{i}" for i in range(1, 29)])
    df.insert(0, "Time", t); df["Amount"] = amt.round(2); df["Class"] = y
    return df

if os.path.exists("creditcard.csv"):
    df = pd.read_csv("creditcard.csv"); SYNTHETIC = False
else:
    df = make_synthetic(); SYNTHETIC = True
    print("WARNING: creditcard.csv not found - using SYNTHETIC data. Results are not real.")
print(df.shape); print("Nulls:", int(df.isnull().sum().sum()), "| Duplicates:", int(df.duplicated().sum()))
df.head()

## 2. Class imbalance: what % of transactions are fraud?

In [ ]:
counts = df["Class"].value_counts()
pct = df["Class"].value_counts(normalize=True) * 100
print(f"Legitimate: {counts[0]:,} ({pct[0]:.3f}%)\nFraud: {counts[1]:,} ({pct[1]:.3f}%)")
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=["Legitimate", "Fraud"], y=counts.values, palette=["#4C78A8", "#E45756"], ax=ax)
ax.set_yscale("log"); ax.set_ylabel("Transactions (log scale)"); ax.set_title("Class Distribution")
for i, v in enumerate(counts.values): ax.text(i, v, f"{v:,}\n({pct[i]:.2f}%)", ha="center", va="bottom")
save("01_class_distribution")

## 3. EDA: transaction amounts and time of day

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x="Class", y="Amount", showfliers=False, palette=["#4C78A8", "#E45756"], ax=axes[0])
axes[0].set_xticklabels(["Legitimate", "Fraud"]); axes[0].set_title("Amount by class (outliers hidden)")
for c, col, lab in [(0, "#4C78A8", "Legitimate"), (1, "#E45756", "Fraud")]:
    sns.kdeplot(np.log1p(df.loc[df.Class == c, "Amount"]), ax=axes[1], color=col, label=lab, fill=True, alpha=.3, common_norm=False)
axes[1].set_xlabel("log(1 + Amount)"); axes[1].set_title("Amount distribution (density)"); axes[1].legend()
save("02_amount_distribution")
print(df.groupby("Class")["Amount"].describe().round(2))

In [ ]:
df["Hour"] = ((df["Time"] // 3600) % 24).astype(int)
hr = df.groupby("Hour")["Class"].agg(["sum", "count"]); hr["fraud_rate_%"] = hr["sum"] / hr["count"] * 100
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(hr.index, hr["count"], color="#4C78A8"); axes[0].set_title("Transactions per hour of day"); axes[0].set_xlabel("Hour")
axes[1].bar(hr.index, hr["fraud_rate_%"], color="#E45756"); axes[1].set_title("Fraud rate (%) per hour of day"); axes[1].set_xlabel("Hour")
save("03_time_of_day_analysis")

**Observations:** write what you see in your real run here, e.g. whether fraud amounts differ from legitimate ones, and whether some hours (often late night) show a higher fraud *rate* even when total volume is low.

## 4. Why accuracy is a misleading metric here
With under 1% fraud, a model that **always predicts "legitimate"** is right over 99% of the time yet catches **zero** fraud. Accuracy rewards ignoring the minority class. So we judge models by **precision, recall, F1 and AUC-ROC / PR-AUC**, which focus on the fraud class.

In [ ]:
X = df.drop(columns=["Class", "Time"]); y = df["Class"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RS)   # stratified: fraud in both splits
print("Fraud in train:", int(y_tr.sum()), "| Fraud in test:", int(y_te.sum()))
scaler = StandardScaler().fit(X_tr[["Amount", "Hour"]])
for part in (X_tr, X_te): part[["Amount", "Hour"]] = scaler.transform(part[["Amount", "Hour"]])
always_legit = np.zeros(len(y_te), dtype=int)
print(f"'Always legitimate' baseline -> accuracy {accuracy_score(y_te, always_legit):.4f}, fraud recall {recall_score(y_te, always_legit):.2f}")

## 5. Handling class imbalance
We use **`class_weight='balanced'`**, which makes each fraud case count more in the loss. This avoids creating synthetic rows. SMOTE is shown as an optional extra (needs `pip install imbalanced-learn`); it must be applied **only to the training set**.

In [ ]:
try:
    from imblearn.over_sampling import SMOTE
    X_sm, y_sm = SMOTE(random_state=RS).fit_resample(X_tr, y_tr)
    print("SMOTE available. Train class counts after SMOTE:", dict(pd.Series(y_sm).value_counts()))
except ImportError:
    print("imbalanced-learn not installed -> skipping SMOTE; using class_weight='balanced'.")

## 6. Train models: Logistic Regression + Random Forest

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RS),
    "Random Forest": RandomForestClassifier(n_estimators=150, class_weight="balanced_subsample", n_jobs=-1, random_state=RS),
}
res, probs, preds = [], {}, {}
for name, m in models.items():
    m.fit(X_tr, y_tr)
    p = m.predict_proba(X_te)[:, 1]; yp = (p >= 0.5).astype(int)
    probs[name], preds[name] = p, yp
    res.append({"Model": name, "Precision": precision_score(y_te, yp), "Recall": recall_score(y_te, yp),
                "F1": f1_score(y_te, yp), "AUC-ROC": roc_auc_score(y_te, p), "PR-AUC": average_precision_score(y_te, p)})
    print(f"\n=== {name} ===\n", classification_report(y_te, yp, target_names=["Legit", "Fraud"], digits=3))
results = pd.DataFrame(res).set_index("Model").round(4); results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, yp) in zip(axes, preds.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_te, yp), display_labels=["Legit", "Fraud"]).plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(name); ax.grid(False)
save("04_confusion_matrices")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
for name, p in probs.items():
    fpr, tpr, _ = roc_curve(y_te, p); ax.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc_score(y_te, p):.3f})")
ax.plot([0, 1], [0, 1], "k--", label="Random guess"); ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate (Recall)")
ax.set_title("ROC Curve"); ax.legend(loc="lower right")
save("05_roc_curve")

fig, ax = plt.subplots(figsize=(6.5, 5))
for name, p in probs.items():
    pr, rc, _ = precision_recall_curve(y_te, p); ax.plot(rc, pr, label=f"{name} (AP = {average_precision_score(y_te, p):.3f})")
ax.axhline(y_te.mean(), color="k", ls="--", label="No-skill baseline")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_title("Precision-Recall Curve"); ax.legend()
save("06_precision_recall_curve")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4)); results.plot(kind="bar", ax=ax, rot=0, colormap="viridis")
ax.set_ylim(0, 1.05); ax.set_title("Model comparison"); ax.legend(loc="lower right", ncol=2)
save("07_model_comparison")

## 7. Which metric matters most? Recall vs. precision
- **Recall** = share of real frauds caught. A missed fraud (false negative) costs the full transaction amount plus chargebacks and lost trust.
- **Precision** = share of alerts that are real fraud. A false alarm (false positive) costs analyst time and annoys customers by blocking legitimate purchases.

Recall is usually prioritised, but not at any price: too many false alarms make the system unusable. The best practice is to **tune the decision threshold** to the business's cost balance, and compare models with **PR-AUC**, which is more informative than ROC-AUC under heavy imbalance.

## 8. Feature importance / coefficients

In [ ]:
rf = models["Random Forest"]; lr = models["Logistic Regression"]
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values().tail(15)
coef = pd.Series(lr.coef_[0], index=X.columns).reindex(pd.Series(lr.coef_[0], index=X.columns).abs().sort_values().tail(15).index)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
imp.plot(kind="barh", ax=axes[0], color="#4C78A8"); axes[0].set_title("Random Forest: top 15 feature importances")
coef.plot(kind="barh", ax=axes[1], color=["#E45756" if v > 0 else "#4C78A8" for v in coef]); axes[1].set_title("Logistic Regression: top 15 coefficients\n(red = pushes toward fraud)")
save("08_feature_importance")

## 9. Scalability: 1 million transactions per hour
1 million/hour is roughly **280 transactions per second**. Scoring is not the hard part: a trained tree ensemble or logistic regression scores a row in well under a millisecond. The challenges are elsewhere:
1. **Latency:** scoring must finish in milliseconds in the payment path, so use a lightweight model, precompute features, and consider a fast first-stage model with a heavier second stage only for borderline cases.
2. **Streaming architecture:** consume events from a queue (e.g. Kafka), score in stateless horizontally scaled workers, and keep per-customer features (recent spend, velocity) in a low-latency store such as Redis.
3. **Concept drift:** fraudsters adapt, so monitor precision/recall, retrain regularly, and use delayed fraud labels (chargebacks arrive weeks later).
4. **Imbalance at scale:** keep stratified sampling for training, and review a stream of flagged cases to get fresh labels.
5. **Human-in-the-loop:** at this volume even 0.1% false positives is about 1,000 alerts per hour, so tune the threshold to analyst capacity.

## 10. Conclusion
*(Fill in from your real-data run.)* Summarise: fraud share, why accuracy fails, which model gave the best recall/precision/PR-AUC, the most influential features, and the threshold you would deploy.

In [ ]:
print("Using synthetic data" if SYNTHETIC else "Using real creditcard.csv"); print(results.to_string())